In [133]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [134]:
# Конструиране на Graph Autoencoder
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE

class Encoder(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, 64)
        self.conv2 = GCNConv(64, 32)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

encoder = Encoder(dataset.num_features)

model = GAE(encoder)

In [135]:
# Създаване на модела
encoder = Encoder(dataset.num_features)
model = GAE(encoder)

In [136]:
# Инициализиране на оптимизатора
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01)

In [137]:
# Създаване на тренировъчен и тестов набор от данни
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    add_negative_train_samples=True)

train_data, val_data, test_data = transform(data)

In [138]:
# Обучение
import time

def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(
        train_data.x,
        train_data.edge_index)
    loss = model.recon_loss(
        z,
        train_data.edge_label_index)
    loss.backward()
    optimizer.step()

    return loss.item()

start = time.time()
for epoch in range(1, 201):
    loss = train()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}   Loss: {loss:.4f}")

training_time = time.time() - start

Epoch  20   Loss: 1.2976
Epoch  40   Loss: 1.2133
Epoch  60   Loss: 1.1736
Epoch  80   Loss: 1.1216
Epoch 100   Loss: 1.0842
Epoch 120   Loss: 1.0467
Epoch 140   Loss: 1.0279
Epoch 160   Loss: 1.0091
Epoch 180   Loss: 1.0030
Epoch 200   Loss: 0.9890


In [139]:
# Оценяване
model.eval()

with torch.no_grad():
    z = model.encode(
        test_data.x,
        test_data.edge_index
    )

reconstruction_loss = model.recon_loss(
    z,
    test_data.edge_label_index
).item()

pos_edge_index = test_data.edge_label_index[
    :, test_data.edge_label == 1
]

neg_edge_index = test_data.edge_label_index[
    :, test_data.edge_label == 0]

auc, ap = model.test(
    z,
    pos_edge_index,
    neg_edge_index)

In [140]:
# Извеждане на резултатите
print(f"Reconstruction Loss: {reconstruction_loss:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Average Precision: {ap:.4f}")
print(f"Training time: {training_time:.2f} s")
print(f"Embeddings shape: {z.shape}")

Reconstruction Loss: 1.3042
AUC: 0.8464
Average Precision: 0.8651
Training time: 7.43 s
Embeddings shape: torch.Size([2708, 32])


In [141]:
# Използване на модела за прогнозиране на връзки
model.eval()

with torch.no_grad():
    z = model.encode(data.x, data.edge_index)

edge = torch.tensor([[10],
                     [25]])

prob = model.decoder(
    z,
    edge,
    sigmoid=True
)

print(f"Probability: {prob.item():.4f}")

Probability: 0.6949
